# GPU が使えるか確認し、メモリを解放する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/01_colab_tips/colab_gpu.ipynb)

NLP クックブック 01-02 ／ Google Colab の小技

GPU ランタイムになっているか、メモリがどれくらい空いているかを確認したい。`CUDA out of memory` から復帰したい。

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("GPU が使える:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"空き {free / 1e9:.1f} GB / 全体 {total / 1e9:.1f} GB")
device = "cuda" if torch.cuda.is_available() else "cpu"   # GPU がなくても動くコードにしておく

PyTorch: 2.15.0+cpu
GPU が使える: False


In [2]:
import gc

x = torch.zeros(256, 1024, 1024, device=device)   # 約 1GB を確保
print(f"確保後: {torch.cuda.memory_allocated() / 1e9:.2f} GB" if device == "cuda" else "CPU で実行中")

del x                      # 1. 参照を消す (モデルやテンソルを指す変数をすべて)
gc.collect()               # 2. Python のガベージコレクションを走らせる
torch.cuda.empty_cache()   # 3. PyTorch がキャッシュしている GPU メモリを返す
print(f"解放後: {torch.cuda.memory_allocated() / 1e9:.2f} GB" if device == "cuda" else "(GPU なし)")

CPU で実行中
(GPU なし)


In [3]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv 2>/dev/null || echo "GPU なし (ランタイム → ランタイムのタイプを変更 → T4 GPU)"

GPU なし (ランタイム → ランタイムのタイプを変更 → T4 GPU)


- ノートブックでは、直前の出力 (`_`) や例外のトレースバックがテンソルへの参照を握っていて解放されないことがある。どうしても空かなければ、ランタイムを再起動するのが早い。
- メモリ不足のときの対処: バッチサイズを下げる → 系列長 (`max_length`) を下げる → 半精度 (`fp16` / `torch_dtype=torch.float16`) → 勾配累積 → 量子化、の順に試す。
- 推論では `torch.inference_mode()` を忘れない。勾配用の中間結果を保持してメモリを浪費する。